# Claude 3와 MongoDB로 RAG 시스템 구축하기


이 tutorial은 Venture capital tech Analyst 역할을 맡도록 prompting한 chatbot을 구현합니다. Chatbot은 기술 뉴스 기사 모음을 지식원으로 사용하는 naive RAG 시스템입니다.
이 notebook은 다음을 다룹니다:

1. 필요한 라이브러리 설치부터 MongoDB database 설정까지 개발 환경 설정에 대한 종합적인 tutorial 따라가기.
2. Vector search index 생성과 ingestion 및 query 처리를 위한 데이터 준비 등 효율적인 데이터 처리 방법 학습.
3. Database에서 retrieval한 context 정보에 기반하여 정확한 응답을 생성하기 위해 RAG 시스템에서 Claude 3 모델을 사용하는 방법 이해.


다음이 필요합니다:
- Claude API Key
- VoyageAI API Key
- Hugging Face Access Token

## 1단계: 라이브러리 설치, 데이터 로드 및 준비


구현 코드에서 사용하는 도구와 라이브러리에 대한 간략한 설명은 다음과 같습니다:
- anthropic: 최첨단 언어 모델에 접근할 수 있는 Anthropic의 공식 Python 라이브러리입니다. 텍스트와 이미지를 이해할 수 있는 Claude 3 제품군 모델에 대한 접근을 제공합니다.
- datasets: Hugging Face 생태계의 일부인 라이브러리입니다. 'datasets'를 설치하면 machine learning 모델의 학습 및 fine-tuning이나 성능 benchmark에 필수적인, 전처리되어 바로 사용할 수 있는 여러 dataset에 접근할 수 있습니다.
- pandas: 데이터 조작, 처리 및 분석을 위한 견고한 자료 구조와 method를 제공하는 data science 라이브러리입니다.
- voyageai: VoyageAI의 embedding 모델군에 접근하는 공식 Python client 라이브러리입니다.
- pymongo: PyMongo는 MongoDB용 Python toolkit입니다. MongoDB database와 상호 작용할 수 있게 합니다.



In [ ]:
%pip install pymongo datasets pandas anthropic voyageai

아래 코드 snippet은 다음 단계를 실행합니다:
1. 필요한 라이브러리 import:
- 운영 체제와 상호 작용하는 `os`,
- HTTP 요청을 보내는 `requests`,
- Byte object를 메모리의 파일처럼 처리하는 io module의 `BytesIO`,
- 데이터 조작 및 분석을 위한 `pandas`(pd로 사용), 그리고
- Google Colab secrets에 저장된 환경 변수에 접근할 수 있게 하는 google.colab의 `userdata`.
2. 함수 정의: `download_and_combine_parquet_files` 함수를 두 parameter로 정의합니다:
- `parquet_file_urls`: 문자열 URL 목록이며, 각 URL은 tech-news-embedding dataset의 일부를 담은 Parquet 파일을 가리킵니다.
- `hf_token`은 Hugging Face 인증 token을 나타내는 문자열입니다. Access token은 [Hugging Face platform](https://huggingface.co/docs/hub/en/security-tokens#:~:text=To%20create%20an%20access%20token,you%27re%20ready%20to%20go!)에서 생성하거나 복사할 수 있습니다
3. Parquet 파일 다운로드 및 읽기: 함수는 parquet_file_urls의 각 URL을 순회합니다. 각 URL에 대해 다음을 수행합니다:
- requests.get method로 URL과 인증 header를 전달하여 GET 요청을 보냅니다.
- 요청 성공을 나타내는 response status code 200 (OK)인지 확인합니다.
- 성공하면 응답 내용을 BytesIO object로 읽고(메모리의 파일처럼 처리하기 위해), pandas.read_parquet으로 해당 object의 Parquet 파일을 Pandas DataFrame으로 읽습니다.
- DataFrame을 all_dataframes 목록에 추가합니다.
4. DataFrame 결합: 모든 Parquet 파일을 다운로드하여 DataFrame으로 읽은 뒤 `all_dataframes`가 비어 있지 않은지 확인합니다. 처리할 DataFrame이 있으면 pd.concat으로 모두 하나의 DataFrame으로 결합하고, ignore_index=True로 새로 결합된 DataFrame의 index를 재설정합니다. 이 결합된 DataFrame이 `download_and_combine_parquet_files` 함수의 전체 과정 output입니다.

In [ ]:
from io import BytesIO

import pandas as pd
import requests
from google.colab import userdata


def download_and_combine_parquet_files(parquet_file_urls, hf_token):
    """
    주어진 Hugging Face token으로 제공된 URL에서 Parquet 파일을 다운로드하고,
    결합된 DataFrame을 반환합니다.

    Parameters:
    - parquet_file_urls: 문자열 목록, Parquet 파일의 URL.
    - hf_token: 문자열, Hugging Face 인증 token.

    Returns:
    - combined_df: 모든 Parquet 파일의 결합된 데이터를 담은 pandas DataFrame.
    """
    headers = {"Authorization": f"Bearer {hf_token}"}
    all_dataframes = []

    for parquet_file_url in parquet_file_urls:
        response = requests.get(parquet_file_url, headers=headers, timeout=60)
        if response.status_code == 200:
            parquet_bytes = BytesIO(response.content)
            df = pd.read_parquet(parquet_bytes)
            all_dataframes.append(df)
        else:
            print(
                f"Failed to download Parquet file from {parquet_file_url}: {response.status_code}"
            )

    if all_dataframes:
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        return combined_df
    else:
        print("No dataframes to concatenate.")
        return None

이 tutorial에 필요한 Parquet 파일 목록은 다음과 같습니다. 전체 파일 목록은 [여기](https://huggingface.co/datasets/MongoDB/tech-news-embeddings/tree/refs%2Fconvert%2Fparquet/default/train)에 있습니다. 각 Parquet 파일은 약 45,000개의 데이터 포인트를 나타냅니다.

아래 코드 snippet에서는 tech-news-embeddings dataset의 일부를 하나의 DataFrame으로 묶고 `combined_df` 변수에 할당합니다.


In [3]:
# 더 많은 데이터를 로드하려면 아래 링크의 주석을 해제하세요
# 전체 데이터 목록: https://huggingface.co/datasets/MongoDB/tech-news-embeddings/tree/refs%2Fconvert%2Fparquet/default/train
parquet_files = [
    "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0000.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0001.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0002.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0003.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0004.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0005.parquet",
]

hf_token = userdata.get("HF_TOKEN")
combined_df = download_and_combine_parquet_files(parquet_files, hf_token)

데이터 준비의 마지막 단계로, 아래 코드 snippet은 묶인 dataset에서 `_id` column을 제거하는 단계를 보여줍니다. 이 tutorial의 후속 단계에는 필요하지 않기 때문입니다. 또한 ingestion 중 호환되지 않는 data type 관련 오류를 방지하기 위해 각 데이터 포인트의 embedding column 데이터를 numpy array에서 Python list로 변환합니다. 


In [4]:
# 초기 dataset에서 _id column 제거
combined_df = combined_df.drop(columns=["_id"])

# VoyageAI embedding 모델로 새 embedding을 생성할 것이므로 초기 embedding column 제거
combined_df = combined_df.drop(columns=["embedding"])

In [ ]:
combined_df.head()

In [ ]:
# VoyageAI API rate limit 때문에 이 데모에서 사용하는 문서 수를 500개로 제한
# VoyageAI rate limit에 대한 자세한 내용: https://docs.voyageai.com/docs/rate-limits
max_documents = 500

if len(combined_df) > max_documents:
    combined_df = combined_df[:max_documents]

In [ ]:
import voyageai

vo = voyageai.Client(api_key=userdata.get("VOYAGE_API_KEY"))


def get_embedding(text: str) -> list[float]:
    if not text.strip():
        print("Attempted to get embedding for empty text.")
        return []

    embedding = vo.embed(text, model="voyage-large-2", input_type="document")

    return embedding.embeddings[0]


combined_df["embedding"] = combined_df["description"].apply(get_embedding)

combined_df.head()

## 2단계: Database 및 collection 생성

**새 MongoDB database를 만들려면 database cluster를 설정하세요:**
1. [무료 MongoDB Atlas 계정](https://www.mongodb.com/cloud/atlas/register?utm_campaign=devrel&utm_source=community&utm_medium=cta&utm_content=Partner%20Cookbook&utm_term=richmond.alake)에 가입하거나, 기존 사용자는 [MongoDB Atlas에 로그인](https://account.mongodb.com/account/login?utm_campaign=devrel&utm_source=community&utm_medium=cta&utm_content=Partner%20Cookbook&utm_term=richmond.alake)하세요
2. 왼쪽 pane에서 “Database”를 선택하면 기존 cluster의 deployment specification이 있는 Database Deployment 페이지로 이동합니다. "+Create" 버튼을 클릭하여 새 database cluster를 생성하세요.
3. Database cluster 설정 및 URI를 얻는 데 도움이 필요하면 MongoDB cluster 설정 및 connection string 발급 가이드를 참조하세요.
참고: Proof of concept를 만들 때 Python host의 IP 또는 모든 IP를 위한 0.0.0.0/0을 whitelist에 추가하는 것을 잊지 마세요.
4. Cluster를 생성하고 배포하는 데 성공하면 ‘Database Deployment’ 페이지에서 접근할 수 있습니다.
5. Cluster의 “Connect” 버튼을 클릭하면 다양한 언어 driver를 통해 cluster 연결을 설정하는 옵션을 볼 수 있습니다.
6. 이 tutorial에서는 cluster의 URI(unique resource identifier)만 필요합니다. URI를 가져와 Google Colabs Secrets 환경의 MONGO_URI라는 변수에 복사하거나 .env 파일 또는 이에 해당하는 곳에 저장하세요.


Cluster를 생성한 뒤 cluster 페이지로 이동하여 + Create Database를 클릭하고 MongoDB Atlas cluster 내에 database와 collection을 만드세요. 
Database 이름은 `tech_news`, collection 이름은 `hacker_noon_tech_news`입니다.



## 3단계: Vector search index 생성

이 시점에는 cluster, database 및 collection이 생성되어 있습니다.

이 절의 단계는 chatbot에 입력된 query로 hacker_noon_tech_news collection의 record를 대상으로 vector search를 수행할 수 있도록 보장하는 데 중요합니다. 이 단계의 목표는 vector search index 생성입니다. 이를 위해 공식 [vector search index 생성 가이드](https://www.mongodb.com/docs/atlas/atlas-vector-search/create-index/)를 참조하세요.

MongoDB Atlas의 JSON editor로 vector search index를 생성할 때 index 이름을 vector_index로 지정하고 vector search index 정의를 다음과 같이 설정하세요:

```
{
 "fields": [{
     "numDimensions": 1536,
     "path": "embedding",
     "similarity": "cosine",
     "type": "vector"
   }]
}

```

## 4단계: 데이터 ingestion

이전 단계에서 만든 MongoDB database에 데이터를 ingestion하려면 다음 작업을 수행해야 합니다:
- Database 및 collection에 연결
- Collection에서 기존 record 모두 제거
- Ingestion 전에 dataset의 Pandas DataFrame을 dictionary로 변환
- Batch 작업으로 dictionary를 MongoDB에 ingestion

이 tutorial에는 cluster의 URI(unique resource identifier)가 필요합니다. URI를 가져와 Google Colab Secrets 환경의 MONGO_URI라는 변수에 복사하거나 .env 파일 또는 이에 해당하는 곳에 저장하세요.


In [6]:
import pymongo
from google.colab import userdata


def get_mongo_client(mongo_uri):
    """MongoDB에 연결합니다."""
    try:
        client = pymongo.MongoClient(mongo_uri)
        print("Connection to MongoDB successful")
        return client
    except pymongo.errors.ConnectionFailure as e:
        print(f"Connection failed: {e}")
        return None


mongo_uri = userdata.get("MONGO_URI")
if not mongo_uri:
    print("MONGO_URI not set in environment variables")

mongo_client = get_mongo_client(mongo_uri)

DB_NAME = "tech_news"
COLLECTION_NAME = "hacker_noon_tech_news"

db = mongo_client[DB_NAME]
collection = db[COLLECTION_NAME]

Connection to MongoDB successful


In [7]:
# 새로운 collection으로 작업하도록
# collection의 기존 record 모두 삭제
collection.delete_many({})

DeleteResult({'n': 228012, 'electionId': ObjectId('7fffffff000000000000000e'), 'opTime': {'ts': Timestamp(1709660559, 7341), 't': 14}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1709660559, 7341), 'signature': {'hash': b'jT\xf1\xb4\xa9\xd3\xe3suu\x03`\x15(}\x8f\x00\x9f\xe9\x8a', 'keyId': 7320226449804230661}}, 'operationTime': Timestamp(1709660559, 7341)}, acknowledged=True)

In [ ]:
# 데이터 ingestion
combined_df_json = combined_df.to_dict(orient="records")
collection.insert_many(combined_df_json)

## 5단계: Vector Search

이 절에서는 chatbot에 입력되는 user query를 받는 vector search custom 함수를 생성하는 방법을 보여줍니다. 함수는 두 번째 parameter인 `collection`도 받으며, 이는 vector search를 수행할 대상 record를 담은 database collection을 가리킵니다.

`vector_search` 함수는 MongoDB aggregation pipeline에 정의된 일련의 작업으로 vector search 결과를 생성합니다. Pipeline에는 `$vectorSearch` 및 `$project` stage가 포함되며 user query의 vector embedding에 기반하여 query를 수행합니다. 그런 다음 후속 과정에 불필요한 record attribute를 제외하여 결과 형식을 정리합니다.

아래 코드 snippet은 영화에 대한 semantic search를 위해 다음 작업을 수행합니다:
1. User query 문자열과 MongoDB collection을 input으로 받고 vector similarity search에 기반하여 query와 일치하는 문서 목록을 반환하는 `vector_search` 함수를 정의합니다.
2. 이전에 정의한 `get_embedding` 함수를 호출하여 user query의 embedding을 생성합니다. 이 함수는 query 문자열을 vector 표현으로 변환합니다.
3. MongoDB의 aggregate 함수를 위한 pipeline을 구성하며 주요 stage인 `$vectorSearch`와 `$project`를 포함합니다.
4. `$vectorSearch` stage는 실제 vector search를 수행합니다. index field는 vector search에 사용할 vector index를 지정하며, 이전 단계의 vector search index 정의에 입력한 이름과 일치해야 합니다. queryVector field는 user query의 embedding 표현을 받습니다. path field는 embedding을 담은 문서 field에 해당합니다. `numCandidates`는 고려할 후보 문서 수와 반환할 결과 수의 제한을 지정합니다.
5. $project stage는 _id와 `embedding` field를 제외하도록 결과 형식을 정리합니다.
6. aggregate는 정의된 pipeline을 실행하여 vector search 결과를 얻습니다. 마지막 작업은 database가 반환한 cursor를 목록으로 변환합니다.

In [9]:
def vector_search(user_query, collection):
    """
    User query에 기반하여 MongoDB collection에서 vector search를 수행합니다.

    Args:
    user_query (str): User query 문자열.
    collection (MongoCollection): 검색할 MongoDB collection.

    Returns:
    list: 일치하는 문서 목록.
    """

    # User query의 embedding 생성
    query_embedding = get_embedding(user_query)

    if query_embedding is None:
        return "Invalid query or embedding generation failed."

    # Vector search pipeline 정의
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 150,  # 고려할 일치 후보 수
                "limit": 5,  # 상위 5개 일치 결과 반환
            }
        },
        {
            "$project": {
                "_id": 0,  # _id field 제외
                "embedding": 0,  # embedding field 제외
                "score": {
                    "$meta": "vectorSearchScore"  # 검색 score 포함
                },
            }
        },
    ]

    # 검색 실행
    results = collection.aggregate(pipeline)
    return list(results)

## 6단계: Claude 3 모델로 user query 처리

Tutorial의 마지막 절에서는 다음 작업 순서를 설명합니다:

- 문자열 형태의 user query 수신.
- VoyageAI embedding 모델로 user query의 embedding 생성.
- RAG 시스템의 base model로 사용할 Anthropic의 Claude, 구체적으로 `claude-opus-4-8` 모델 로드.
- User query의 embedding으로 vector search를 실행하여 knowledge base에서 관련 정보 조회. 이는 base model에 추가 context를 제공합니다.
- User query와 수집한 추가 정보를 모두 base model에 제출하여 응답 생성.


중요한 점은 user query embedding의 dimension이 MongoDB Atlas의 vector search index 정의에 설정된 dimension과 일치해야 한다는 것입니다.

이 절의 다음 단계는 anthropic 라이브러리를 import하고 client를 로드하여 message 처리 및 Claude 모델 접근을 위한 anthropic method에 접근하는 것입니다. [Anthropic 공식 웹사이트](https://console.anthropic.com/settings/keys)의 설정 페이지에서 Claude API key를 발급받으세요.


In [11]:
import anthropic

client = anthropic.Client(api_key=userdata.get("ANTHROPIC_API_KEY"))

아래 코드 snippet의 작업에 대한 더 자세한 설명은 다음과 같습니다:

1. Vector Search 실행: 함수는 user query와 지정된 collection을 argument로 `vector_search`를 호출하며 시작합니다. 이를 통해 collection에서 검색을 수행하며, vector embedding을 활용하여 query와 관련된 정보를 찾습니다.
2. 검색 결과 취합: 검색 정보를 모으기 위해 `search_result`를 빈 문자열로 초기화합니다. `vector_search` 함수가 반환한 결과를 순회하고 각 항목의 세부 정보(제목, 회사 이름, URL, 발행일, 기사 URL 및 설명)를 사람이 읽기 쉬운 문자열로 정리하며, 각 항목 끝에 newline 문자 
을 추가하여 search_result에 붙입니다.
3. Anthropic Client로 응답 생성: 그런 다음 함수는 Claude API에 보낼 요청을 구성합니다(client object를 사용하며, 이전에 생성한 anthropic.Client class의 instance로 추정됩니다). 다음을 지정합니다:
- 사용할 모델(`claude-opus-4-8`)은 Claude Opus 모델 버전을 나타냅니다.
- 생성되는 응답의 최대 token 제한(max_tokens=1024).
- System 설명은 기술 기업의 기사와 정보에 접근할 수 있는 "Venture Capital Tech Analyst"로 행동하고 이 context로 조언하도록 모델을 안내합니다.
- 모델이 처리할 실제 message는 user query와 취합한 검색 결과를 context로 결합합니다.
4. 생성된 응답 및 검색 결과 반환: 응답 content의 첫 항목에서 응답 텍스트를 추출하여 취합한 검색 결과와 함께 반환합니다.


In [12]:
def handle_user_query(query, collection):
    get_knowledge = vector_search(query, collection)

    search_result = ""
    for result in get_knowledge:
        search_result += (
            f"Title: {result.get('title', 'N/A')}, "
            f"Company Name: {result.get('companyName', 'N/A')}, "
            f"Company URL: {result.get('companyUrl', 'N/A')}, "
            f"Date Published: {result.get('published_at', 'N/A')}, "
            f"Article URL: {result.get('url', 'N/A')}, "
            f"Description: {result.get('description', 'N/A')}, \n"
        )

    response = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=1024,
        system="You are Venture Captital Tech Analyst with access to some tech company articles and information. You use the information you are given to provide advice.",
        messages=[
            {
                "role": "user",
                "content": "Answer this user query: "
                + query
                + " with the following context: "
                + search_result,
            }
        ],
    )

    return (response.content[0].text), search_result

이 tutorial의 마지막 단계는 query를 초기화하고 `handle_user_query` 함수에 전달한 뒤 반환된 응답을 출력하는 것입니다.

In [13]:
# 출처 retrieval과 함께 query 수행
query = "Give me the best tech stock to invest in and tell me why"
response, source_information = handle_user_query(query, collection)

print(f"Response: {response}")
print(f"\\nSource Information: \\n{source_information}")

Response: Based on the information provided in the article titles and descriptions, Alibaba Group Holding Limited appears to be a top technology stock pick for 2023 according to renowned investor Ray Dalio. The article "Top 10 Technology Stocks to Buy in 2023 According to Ray Dalio" suggests that Alibaba is one of Dalio's favored tech investments for the year.

As a venture capital tech analyst, I would recommend considering an investment in Alibaba for the following reasons:

1. Endorsement from a respected investor: Ray Dalio, known for his successful investment strategies, has included Alibaba in his top 10 technology stock picks for 2023. His backing lends credibility to the investment potential of the company.

2. Strong market position: Alibaba is a leading e-commerce company in China with a significant market share. It has a diversified business model spanning e-commerce, cloud computing, digital media, and entertainment.

3. Growth potential: With China's large and growing midd